In [151]:
import os
import rootutils

from tqdm.notebook import tqdm

rootutils.setup_root(os.path.abspath('./'), indicator=".project-root", pythonpath=True, dotenv=True, cwd=True)

# auto-loading of imports from outside scripts
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [152]:
import pandas as pd
import re
import numpy as np

import optuna
from lightning import seed_everything

seed_everything(42, verbose=False)
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [153]:
df_coord_numbs = pd.read_csv("data_cod/coord_numbs.csv")
df_coord_numbs.rename(columns={"smiles": "can_smiles"}, inplace=True)

In [154]:
# your allowed elements
organic = {"C", "H", "O", "N", "S", "F", "Cl", "Br", "P"}

# regex to pull out element symbols (Cl, Br or any capital letter + optional lowercase)
_pattern = re.compile(r'Cl|Br|[A-Z][a-z]?')

def smiles_only_organic(smiles: str) -> bool:
    """Return True if every element token in a SMILES string is in our organic set."""
    tokens = _pattern.findall(smiles)
    return all(tok in organic for tok in tokens)

def col_name_ok(col: str) -> bool:
    """
    Return True if, when splitting the column name on dash/–,
    every part is one of our organic elements.
    """
    parts = re.split(r'[–-]', col)
    return all(part in organic for part in parts)

# 1) Filter rows by can_smiles
df_rows = df_coord_numbs[df_coord_numbs['can_smiles'].apply(smiles_only_organic)]

# 2) Build list of columns to keep (and also keep can_smiles itself)
keep_cols = ['id', 'can_smiles'] + [c for c in df_rows.columns if col_name_ok(c)]

# 3) Slice down to just those columns
df_coord_numbs = df_rows[keep_cols]

#### Leave only organic crystalls:
`C, H, O, N, S, F, Cl, Br, P`

In [155]:
organic_elements = ["C", "H", "O", "N", "S", "F", "Cl", "Br", "P"]

---
## Merging with temperatures:

In [156]:
# Drop rows and columns that contain only zeros:

rows_only_zeros = df_coord_numbs[(df_coord_numbs == 0).all(axis=1)]
columns_only_zeros = df_coord_numbs.loc[:, (df_coord_numbs == 0).all(axis=0)]

df_coord_numbs = df_coord_numbs.drop(columns=columns_only_zeros.columns, index=rows_only_zeros.index)

In [157]:
df_merged_temp = pd.read_csv("data_cod/cod_bradley_merged.csv")
df_merged_temp = df_merged_temp[df_merged_temp["id"].isin(df_coord_numbs["id"])]

# Only bradley, to use part of it for test:
df_bradley_temp = pd.read_csv("data_cod/bradley_with_cif.csv")
df_bradley_temp = df_bradley_temp[df_bradley_temp["id"].isin(df_coord_numbs["id"])]

---
## Creating Train and Test splits:
- 0.2 of Bradley is test, rest is train

In [158]:
TEST_BRADLEY_FRAC = 0.2

In [159]:
df_test = df_bradley_temp.sample(frac=TEST_BRADLEY_FRAC, random_state=42)

df_train = df_merged_temp.drop(index=df_merged_temp[df_merged_temp["id"].isin(df_test["id"])].index)

In [160]:
df_train = pd.merge(df_train, df_coord_numbs, on=["id", "can_smiles"], how="inner")
df_test = pd.merge(df_test, df_coord_numbs, on=["id", "can_smiles"], how="inner")

In [161]:
X_train_cn = df_train.drop(columns=["id", "can_smiles", "T", "cif_path"])
X_train_smiles = pd.DataFrame(df_train["can_smiles"])
y_train = df_train["T"].astype(float)

X_test_cn = df_test.drop(columns=["id", "can_smiles", "T"])
X_test_smiles = pd.DataFrame(df_test["can_smiles"])
y_test = df_test["T"].astype(float)

### Turning smiles into features:

In [162]:
from src.utils import create_or_load_data
from src.avail_descriptors import descriptors_all

In [163]:
data_args = {
    "descriptors": descriptors_all,

    "apply_norm": False,
    
    "create_fingerprints": False,
    "temp_column": False,
}

In [164]:
train_label_dummy = [1] * len(df_train)
X_train_smiles["label"] = train_label_dummy

In [165]:
X_train_smiles, _, _ = create_or_load_data(X_train_smiles[:], data_args, "saved_np_obj_with_embed_molecule/cif_data_train", load_data=True)

Loading data from saved_np_obj_with_embed_molecule/cif_data_train


In [166]:
test_label_dummy = [1] * len(df_test)
X_test_smiles["label"] = test_label_dummy

In [168]:
X_test_smiles, _, _ = create_or_load_data(X_test_smiles, data_args, "saved_np_obj_with_embed_molecule/cif_data_test", load_data=True)

Loading data from saved_np_obj_with_embed_molecule/cif_data_test


---
# Catboost on CN:

In [169]:
import numpy as np

from src.high_level_train_wrap import train_optuna_catboost

In [170]:
best_params = train_optuna_catboost(X_train_cn, y_train, X_test_cn, y_test, n_trials=50, n_jobs=8)

  0%|          | 0/50 [00:00<?, ?it/s]

🏆 New Best Trial 7: R2=0.65497, RMSE=57.57219
🏆 New Best Trial 3: R2=0.68090, RMSE=55.36670
🏆 New Best Trial 9: R2=0.68238, RMSE=55.23794
🏆 New Best Trial 1: R2=0.70312, RMSE=53.40406
🏆 New Best Trial 34: R2=0.70390, RMSE=53.33448


---
# Neural Net on CN:

In [171]:
from argparse import Namespace
import torch

from torch.utils.data import TensorDataset, DataLoader
from src.multitask_nn import RegressionModel

from sklearn.preprocessing import Normalizer

from src.high_level_train_wrap import train_regression_nn

In [172]:
cfg = Namespace(
    project_name="cif_data_coord_numbs",

    batch_size=256,
    lr=1e-3,
    scheduler_gamma=0.995,
    max_epochs=300,

    hid_dim=768,
    depth=5,
    use_residual=True,
    drop=0.2,

    num_workers=0,
    persistent_workers=False,
)

In [173]:
normer = Normalizer()
X_train_cn_norm = normer.fit_transform(X_train_cn.to_numpy())
X_test_cn_norm = normer.transform(X_test_cn.to_numpy())

In [174]:
nn_model_cn = train_regression_nn(
    X_train=X_train_cn_norm,
    y_train=y_train.astype(float).to_numpy(),
    X_val=X_test_cn_norm,
    y_val=y_test.astype(float).to_numpy(),
    cfg=cfg,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | shared    | Sequential        | 811 K  | train
1 | regressor | Sequential        | 263 K  | train
2 | val_mse   | MeanSquaredError  | 0      | train
3 | val_mae   | MeanAbsoluteError | 0      | train
4 | val_r2    | R2Score           | 0      | train
--------------------------------------------------------
1.1 M     Trainable params
0         Non-trainable params
1.1 M     Total params
4.297     Total estimated model params size (MB)
32        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (11) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=50` reached.


---
# Catboost on Molecular features:

In [175]:
best_params = train_optuna_catboost(X_train_smiles, y_train, X_test_smiles, y_test, n_trials=50, n_jobs=8)

  0%|          | 0/50 [00:00<?, ?it/s]

🏆 New Best Trial 0: R2=0.80224, RMSE=43.58728
🏆 New Best Trial 1: R2=0.82245, RMSE=41.30010
🏆 New Best Trial 7: R2=0.83363, RMSE=39.97856
🏆 New Best Trial 8: R2=0.83975, RMSE=39.23631
🏆 New Best Trial 3: R2=0.85067, RMSE=37.87588
🏆 New Best Trial 4: R2=0.86364, RMSE=36.19376
🏆 New Best Trial 42: R2=0.86453, RMSE=36.07486
🏆 New Best Trial 38: R2=0.86742, RMSE=35.68836
🏆 New Best Trial 39: R2=0.86959, RMSE=35.39457


---
# NN on Molecular features:
- First need to normalize features

In [216]:
cfg = Namespace(
    project_name="cif_data_molecular",

    batch_size=256,
    lr=1e-3,
    scheduler_gamma=0.995,
    max_epochs=200,

    hid_dim=768,
    depth=5,
    use_residual=True,
    drop=0.2,

    num_workers=0,
    persistent_workers=False,
)

In [217]:
def drop_nan(X, y):
    mask = ~np.isnan(X).any(axis=1)

    X_clean = X[mask]
    y_clean = y[mask]
    
    return X_clean, y_clean

X_train_smiles_no_nan, y_train_no_nan = drop_nan(X_train_smiles, y_train)

In [218]:
normer = Normalizer()
X_train_smiles_norm_no_nan = normer.fit_transform(X_train_smiles_no_nan)
X_test_smiles_norm = normer.transform(X_test_smiles)

In [219]:
nn_model_smiles = train_regression_nn(
    X_train=X_train_smiles_norm_no_nan,
    y_train=y_train_no_nan.to_numpy(dtype=float),
    X_val=X_test_smiles_norm,
    y_val=y_test.to_numpy(dtype=float),
    cfg=cfg,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type              | Params | Mode 
--------------------------------------------------------
0 | shared    | Sequential        | 2.4 M  | train
1 | regressor | Sequential        | 591 K  | train
2 | val_mse   | MeanSquaredError  | 0      | train
3 | val_mae   | MeanAbsoluteError | 0      | train
4 | val_r2    | R2Score           | 0      | train
--------------------------------------------------------
3.0 M     Trainable params
0         Non-trainable params
3.0 M     Total params
12.144    Total estimated model params size (MB)
38        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.
/home/aleksandr.varlamov/phase-prediction/.venv/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:310: The number of training batches (21) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=200` reached.


---
# Catboost on CN + Molecular features

In [220]:
notna_mask = ~np.isnan(X_train_smiles).any(axis=1)

In [235]:
X_train_concat = np.concatenate([X_train_cn, X_train_smiles], axis=1)
X_train_concat = X_train_concat[notna_mask]
y_train_notna = y_train[notna_mask]

X_test_concat = np.concatenate([X_test_cn, X_test_smiles], axis=1)

In [232]:
# normer = Normalizer()
# X_train_concat_norm = normer.fit_transform(X_train_concat)
# X_test_concat_norm = normer.transform(X_test_concat)

In [ ]:
best_params = train_optuna_catboost(X_train_concat, y_train_notna, X_test_concat, y_test, n_trials=100, n_jobs=16)

  0%|          | 0/100 [00:00<?, ?it/s]

🏆 New Best Trial 29: R2=0.79889, RMSE=43.95408
🏆 New Best Trial 26: R2=0.79948, RMSE=43.89010
🏆 New Best Trial 20: R2=0.80202, RMSE=43.61158
🏆 New Best Trial 24: R2=0.80489, RMSE=43.29401
🏆 New Best Trial 18: R2=0.82444, RMSE=41.06747
🏆 New Best Trial 11: R2=0.83105, RMSE=40.28661
🏆 New Best Trial 6: R2=0.83113, RMSE=40.27791
🏆 New Best Trial 0: R2=0.83218, RMSE=40.15240
🏆 New Best Trial 22: R2=0.83309, RMSE=40.04318
